# 03 Feature Engineering

This notebook creates the **temporal, operational, rolling, risk, and spatial features** used later in the analysis.

Inputs:
- cleaned dataset from `02_data_preparation.ipynb`

Outputs:
- `dublin_bikes_features.parquet`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ALL_PATH = PROCESSED_DIR / 'dublin_bikes_all.parquet'
if not ALL_PATH.exists():
    ALL_PATH = PROCESSED_DIR / 'dublin_bikes_all.csv.gz'

if ALL_PATH.suffix == '.parquet':
    df = pd.read_parquet(ALL_PATH)
else:
    df = pd.read_csv(ALL_PATH)

df['last_reported'] = pd.to_datetime(df['last_reported'], errors='coerce')
df = df.sort_values(['station_id', 'last_reported']).copy()
print(df.shape)

(13513883, 9)


In [2]:
# --------------------------------------------------
# Paths (as per your project structure)
# --------------------------------------------------
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_PATH = PROCESSED_DIR / "dublin_bikes_features.parquet"
# fallback if needed:
if not INPUT_PATH.exists():
    INPUT_PATH = PROCESSED_DIR / "dublin_bikes_all.parquet"

OUTPUT_PARQUET = PROCESSED_DIR / "dublin_bikes_features_with_trend.parquet"
OUTPUT_CSV = PROCESSED_DIR / "dublin_bikes_features_with_trend.csv.gz"


# --------------------------------------------------
# Load dataset
# --------------------------------------------------
df = pd.read_parquet(INPUT_PATH)

# --------------------------------------------------
# Ensure datetime + sorting
# --------------------------------------------------
df["last_reported"] = pd.to_datetime(df["last_reported"])
df = df.sort_values(["station_id", "last_reported"]).copy()

# --------------------------------------------------
# Time-aware trend (correct implementation)
# --------------------------------------------------
df["util_prev"] = df.groupby("station_id")["utilisation"].shift(1)
df["time_prev"] = df.groupby("station_id")["last_reported"].shift(1)

df["delta_util"] = df["utilisation"] - df["util_prev"]
df["delta_min"] = (
    df["last_reported"] - df["time_prev"]
).dt.total_seconds() / 60

df["util_slope"] = df["delta_util"] / df["delta_min"]

# Clean invalid values
df.loc[df["delta_min"] <= 0, "util_slope"] = np.nan
df.loc[df["delta_min"] > 60, "util_slope"] = np.nan

# --------------------------------------------------
# Deviation from rolling mean (very strong feature)
# --------------------------------------------------
df["util_vs_mean"] = df["utilisation"] - df["roll_mean_30min"]

# --------------------------------------------------
# Optional: acceleration (can comment out if not needed)
# --------------------------------------------------
# df["util_acceleration"] = df.groupby("station_id")["util_slope"].diff()

# --------------------------------------------------
# Save updated datasets
# --------------------------------------------------
df.to_parquet(OUTPUT_PARQUET, index=False)
df.to_csv(OUTPUT_CSV, index=False, compression="gzip")

print("Saved:")
print(" -", OUTPUT_PARQUET)
print(" -", OUTPUT_CSV)
print("Shape:", df.shape)

Saved:
 - /home/tbeliere/Projects/DCU/2026-csc1144-dublin-bikes/data/processed/dublin_bikes_features_with_trend.parquet
 - /home/tbeliere/Projects/DCU/2026-csc1144-dublin-bikes/data/processed/dublin_bikes_features_with_trend.csv.gz
Shape: (13513883, 19)


## Temporal and operational features

In [2]:
df['utilisation'] = df['num_bikes_available'] / df['capacity']
df['dock_ratio'] = df['num_docks_available'] / df['capacity']

df['hour_slot'] = df['last_reported'].dt.hour
df['days_of_week'] = df['last_reported'].dt.dayofweek
df['is_weekday'] = df['days_of_week'] < 5

## Risk indicators

In [3]:
# almost empty station (few bikes available)
df['empty_risk'] = df['utilisation'] < 0.05

# almost full station (few docks available)
df['full_risk'] = df['utilisation'] > 0.95

## Time-based rolling features per station

In [ ]:
df = df.set_index('last_reported')

rolling = (
    df.groupby('station_id')['utilisation']
      .rolling('30min', min_periods=1)
)
df["roll_count_30min"] = rolling.count().reset_index(level=0, drop=True)
df['roll_mean_30min'] = rolling.mean().reset_index(level=0, drop=True)
df['roll_std_30min']  = rolling.std(ddof=0).reset_index(level=0, drop=True)

df = df.reset_index()

## Station-level spatial features

In [5]:
# one row per station using averaged coordinates
stations = (
    df.groupby('station_id', as_index=False)
      .agg({
          'lat': 'mean',
          'lon': 'mean',
          'capacity': 'max',
          'name': 'first',
          'address': 'first'
      })
)

# O'Connell Bridge used as approximate city centre
center_lat = 53.3498
center_lon = -6.2603

stations['distance_center'] = np.sqrt(
    (stations['lat'] - center_lat) ** 2 +
    (stations['lon'] - center_lon) ** 2
)


def center_category(d):
    if d < 0.01:
        return 1   # central
    elif d < 0.02:
        return 2   # intermediate
    else:
        return 3   # peripheral


stations['center_index'] = stations['distance_center'].apply(center_category)
stations.head()

,station_id,lat,lon,capacity,name,address,distance_center,center_index
0,1,53.340919,-6.262501,31,CLARENDON ROW,Clarendon Row,0.009149,1
1,2,53.356781,-6.268140,20,BLESSINGTON STREET,Blessington Street,0.010497,2
2,3,53.351185,-6.269859,20,BOLTON STREET,Bolton Street,0.009658,1
3,4,53.346886,-6.272977,20,GREEK STREET,Greek Street,0.013008,2
4,5,53.330673,-6.260178,40,CHARLEMONT PLACE,Charlemont Street,0.019127,2


## Optional OSM-based area type (load if previously computed)

In [12]:
# If a precomputed stations file exists, reuse it instead of rerunning OSM queries.
from pathlib import Path
import osmnx as ox

stations_features_path = PROCESSED_DIR / 'stations_features.parquet'

if stations_features_path.exists():
    stations = pd.read_parquet(stations_features_path)
else:
    stations = (
        df.groupby("station_id", as_index=False)
        .agg({
            "lat": "mean",
            "lon": "mean"
        })
    )

    center_lat = 53.3498
    center_lon = -6.2603

    stations["distance_center"] = np.sqrt(
        (stations["lat"] - center_lat)**2 +
        (stations["lon"] - center_lon)**2
    )

    def center_category(d):
        if d < 0.01:
            return 1
        elif d < 0.02:
            return 2
        else:
            return 3

    stations["center_index"] = stations["distance_center"].apply(center_category)

    def getArea(lat, lon):
        tags = {"landuse": True, "tourism": True}
        gdf = ox.features_from_point((lat, lon), tags=tags, dist=150)

        area = "residential"
        tourism = False
        land = None

        if "landuse" in gdf.columns and gdf["landuse"].notna().any():
            land = gdf["landuse"].value_counts().idxmax()

            if land in ["commercial", "industrial", "education"]:
                area = "work"
            elif land == "retail":
                area = "retail"
            elif land == "park":
                area = "park"

        if ("tourism" in gdf.columns and gdf["tourism"].notna().any()) or land == "park":
            tourism = True

        return [area, tourism]

    stations[["area_type_str", "tourism"]] = stations.apply(
        lambda row: getArea(row["lat"], row["lon"]),
        axis=1,
        result_type="expand"
    )

    stations.to_parquet(stations_features_path, index=False)

## Merge station-level features back to the row-level dataset

In [13]:
df_features = df.merge(
    stations[['station_id', 'center_index', 'area_type_str']],
    on='station_id',
    how='left'
)

feature_cols = [
    'station_id', 'last_reported', 'hour_slot', 'days_of_week', 'is_weekday',
    'roll_mean_30min', 'roll_std_30min', 'utilisation', 'dock_ratio',
    'empty_risk', 'full_risk', 'center_index', 'area_type_str'
]

df_features = df_features[feature_cols].copy()
df_features.head()

,station_id,last_reported,hour_slot,days_of_week,is_weekday,roll_mean_30min,roll_std_30min,utilisation,dock_ratio,empty_risk,full_risk,center_index,area_type_str
0,1,2022-04-27 13:00:02,13,2,True,0.0,0.0,0.0,0.870968,True,False,1,retail
1,1,2022-04-27 13:30:02,13,2,True,0.0,0.0,0.0,0.870968,True,False,1,retail
2,1,2022-04-27 14:00:02,14,2,True,0.0,0.0,0.0,0.870968,True,False,1,retail
3,1,2022-04-27 14:30:02,14,2,True,0.0,0.0,0.0,0.000000,True,False,1,retail
4,1,2022-04-27 15:00:02,15,2,True,0.0,0.0,0.0,0.000000,True,False,1,retail


## Optimise dtypes and save

In [14]:
df_features['station_id'] = df_features['station_id'].astype('int16')
df_features['hour_slot'] = df_features['hour_slot'].astype('int8')
df_features['days_of_week'] = df_features['days_of_week'].astype('int8')
df_features['center_index'] = df_features['center_index'].astype('int8')

df_features['is_weekday'] = df_features['is_weekday'].astype('bool')
df_features['empty_risk'] = df_features['empty_risk'].astype('bool')
df_features['full_risk'] = df_features['full_risk'].astype('bool')

df_features['roll_mean_30min'] = df_features['roll_mean_30min'].astype('float32')
df_features['roll_std_30min'] = df_features['roll_std_30min'].astype('float32')
df_features['utilisation'] = df_features['utilisation'].astype('float32')
df_features['dock_ratio'] = df_features['dock_ratio'].astype('float32')
df_features['area_type_str'] = df_features['area_type_str'].astype('category')

features_parquet = PROCESSED_DIR / 'dublin_bikes_features.parquet'
features_csv_gz = PROCESSED_DIR / 'dublin_bikes_features.csv.gz'

df_features.to_parquet(features_parquet, index=False)
df_features.to_csv(features_csv_gz, index=False, compression='gzip')

print('Saved:', features_parquet)
print('Saved:', features_csv_gz)
df_features.info()

Saved: D:\DCU\Sem2\Data analysis and mining\Project\2026-csc1144-dublin-bikes\data\processed\dublin_bikes_features.parquet
Saved: D:\DCU\Sem2\Data analysis and mining\Project\2026-csc1144-dublin-bikes\data\processed\dublin_bikes_features.csv.gz
<class 'pandas.DataFrame'>
RangeIndex: 13513883 entries, 0 to 13513882
Data columns (total 13 columns):
 #   Column           Dtype         
---  ------           -----         
 0   station_id       int16         
 1   last_reported    datetime64[us]
 2   hour_slot        int8          
 3   days_of_week     int8          
 4   is_weekday       bool          
 5   roll_mean_30min  float32       
 6   roll_std_30min   float32       
 7   utilisation      float32       
 8   dock_ratio       float32       
 9   empty_risk       bool          
 10  full_risk        bool          
 11  center_index     int8          
 12  area_type_str    category      
dtypes: bool(3), category(1), datetime64[us](1), float32(4), int16(1), int8(3)
memory usage: 425